# Advanced example — provision scenario data and classify a failure code

This second example loads the FCC work-order and failure-code datasets into CouchDB, verifies the requested record through the read-only work-order MCP server, and asks Stirrup to return one grounded failure-code description. Loading rebuilds only the `workorder` and `failurecode` databases; it does not modify the source CSV files.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, os, shutil, subprocess, sys

def find_repo(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "servers").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")

REPO = find_repo()
ARTIFACTS = REPO / "artifacts" / "kdd_tutorial"
TRACE_DIR = ARTIFACTS / "trajectories"
LOG_DIR = ARTIFACTS / "logs"
TRACE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("repo:", REPO)
print("python:", sys.version.split()[0])


In [ ]:
# Load environment variables from .env file in the repository root
from dotenv import load_dotenv

def find_repo(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    return None
    
repo = find_repo()
if repo is None:
    raise RuntimeError("Open this notebook from inside the AssetOpsBench repository.")
ENV_FILE = repo / ".env"
if not ENV_FILE.exists():
    raise RuntimeError(f"Missing {ENV_FILE}. Complete 00_environment_setup.ipynb first.")
load_dotenv(ENV_FILE, override=True)
print("environment source:", ENV_FILE)

## 1. Load the repository FCC scenario

The FCC scenario is already stored inside this AssetOpsBench repository at `src/couchdb/scenarios_data/scenario_kdd_fcc`. The notebook validates its existing manifest and CSV files, then loads that scenario into local CouchDB. It does not copy data from another repository or rewrite the scenario files.


In [ ]:
SCENARIO_NAME = "kdd_fcc"
SCENARIO_DIR = REPO / "src" / "couchdb" / "scenarios_data" / "scenario_kdd_fcc"
MANIFEST_PATH = SCENARIO_DIR / "manifest.json"
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"FCC scenario manifest not found: {MANIFEST_PATH}")

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
expected_collections = {"workorder", "failurecode"}
if set(manifest) != expected_collections:
    raise RuntimeError(
        f"Unexpected FCC manifest collections: {sorted(manifest)}; "
        f"expected {sorted(expected_collections)}."
    )
scenario_files = {name: SCENARIO_DIR / relative_path for name, relative_path in manifest.items()}
missing_files = [str(path) for path in scenario_files.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError("FCC scenario data files are missing: " + ", ".join(missing_files))

print("scenario directory:", SCENARIO_DIR)
print(json.dumps({name: str(path) for name, path in scenario_files.items()}, indent=2))


In [ ]:
load_cmd = [
    "uv", "run", "--directory", str(REPO),
    "python", "src/couchdb/init_data.py", SCENARIO_NAME,
]
loader_env = os.environ.copy()
loader_env.pop("VIRTUAL_ENV", None)
loader_env["SCENARIOS_DATA_DIR"] = str(SCENARIO_DIR.parent)
loader_env["DEFAULT_MANIFEST"] = str(
    SCENARIO_DIR.parent / "default" / "manifest.json"
)
print("loader scenarios root:", loader_env["SCENARIOS_DATA_DIR"])
loaded = subprocess.run(
    load_cmd,
    cwd=REPO,
    env=loader_env,
    text=True,
    capture_output=True,
    timeout=120,
)
if loaded.returncode != 0:
    raise RuntimeError("CouchDB scenario load failed:\n" + (loaded.stderr or loaded.stdout))
print(loaded.stdout.strip())


## 2. Verify the exact record through MCP

The work-order number begins with `TST`, but the CSV stores its actual `siteid` as `MAIN`. The notebook discovers that relationship from loaded data instead of guessing that the prefix is a site ID. `AOB_READONLY=1` removes work-order write tools from the MCP server.


In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

MCP_ENV = os.environ.copy()
MCP_ENV["AOB_READONLY"] = "1"

async def call_wo(tool_name, **arguments):
    params = StdioServerParameters(
        command="uv",
        args=["run", "--directory", str(REPO), "wo-mcp-server"],
        cwd=str(REPO),
        env=MCP_ENV,
    )
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
    text = "\n".join(getattr(item, "text", str(item)) for item in result.content)
    try:
        payload = json.loads(text)
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Non-JSON response from {tool_name}: {text[:500]}") from exc
    if isinstance(payload, dict) and payload.get("error"):
        raise RuntimeError(f"{tool_name} failed: {payload['error']}")
    return payload


In [ ]:
WORK_ORDER_NUMBER = "TST-WO00032"
listing = await call_wo("list_workorders", page_size=0, page_num=1)
matches = [
    row for row in listing.get("work_orders", [])
    if str(row.get("wonum", "")) == WORK_ORDER_NUMBER
]
if len(matches) != 1:
    raise RuntimeError(
        f"Expected exactly one {WORK_ORDER_NUMBER} record after loading; found {len(matches)}."
    )
SITE_ID = str(matches[0]["siteid"])
selected_record = await call_wo(
    "get_workorder", site_id=SITE_ID, wonum=WORK_ORDER_NUMBER
)
record = selected_record.get("work_order", selected_record)
print("selected site:", SITE_ID)
print("selected work order:", record.get("wonum"))
print("description:", record.get("description"))
print("failure code:", record.get("failurecode"))


## 3. Build the advanced grounded prompt

The natural maintenance-engineer request is retained as scenario context. Operational constraints underneath it require the exact real tool call, prohibit writes, and enforce the one-line output contract needed for reliable evaluation.


In [ ]:
ALLOWED_DESCRIPTIONS = [
    "Breakdown", "Electrical", "Fail to function", "Leaking", "Low output",
    "Minor in-service problems", "Overheating", "Plugged / choked",
    "Structural deficiency", "Vibration",
]

USER_QUESTION = (
    "I am a maintenance engineer reviewing failure-code assignments on work orders. "
    "Can you pull up work order TST-WO00032. If no failure code was recorded, suggest the "
    "single failure-code description that best fits the work order description from our list "
    "(Breakdown, Electrical, Fail to function, Leaking, Low output, Minor in-service problems, "
    "Overheating, Plugged / choked, Structural deficiency, Vibration). If a failure code is "
    "already recorded, return the existing code description as-is. Output only the failure-code "
    "description in a single line. Do NOT write anything back to the database."
)

QUESTION = f"""{USER_QUESTION}

Execution requirements:
- Call wo__get_workorder with exactly site_id="{SITE_ID}" and wonum="{WORK_ORDER_NUMBER}".
- You must execute the tool; never print, simulate, or assume a tool result.
- If retrieval fails, output exactly NOT FOUND.
- If failurecode contains an identifier, resolve its description using only the read-only wo__get_failure_codes tool.
- If failurecode is empty, select exactly one of these descriptions based only on the retrieved description:
{chr(10).join(ALLOWED_DESCRIPTIONS)}
- Do not call any write tool.
- After retrieval, respond with exactly the selected description (or NOT FOUND), with no explanation, quotes, Markdown, punctuation, or additional text.
- Call finish with reason set to exactly that same description.
"""
print(QUESTION)


## 4. Run Stirrup and persist the trajectory


In [ ]:
assert shutil.which("uv"), "Install uv first."
MODEL_ID = (os.getenv("KDD_MODEL_ID") or "").strip()
if not MODEL_ID:
    raise RuntimeError(f"Set KDD_MODEL_ID in {ENV_FILE}.")
if MODEL_ID.startswith("watsonx/"):
    required_credentials = ["WATSONX_APIKEY", "WATSONX_PROJECT_ID"]
elif MODEL_ID.startswith("tokenrouter/"):
    required_credentials = ["TOKENROUTER_API_KEY", "TOKENROUTER_BASE_URL"]
elif MODEL_ID.startswith("litellm_proxy/"):
    required_credentials = ["LITELLM_API_KEY", "LITELLM_BASE_URL"]
else:
    raise RuntimeError(f"Unsupported KDD_MODEL_ID route: {MODEL_ID}")
missing_credentials = [name for name in required_credentials if not os.getenv(name)]
if missing_credentials:
    raise RuntimeError("Configure credentials and restart the kernel: " + ", ".join(missing_credentials))
print("agent framework: Stirrup")
print("model:", MODEL_ID)
print("credentials: ready")


In [ ]:
AGENT_TIMEOUT_SECONDS = int(os.getenv("KDD_AGENT_TIMEOUT_SECONDS", "240"))
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID = f"kdd-stirrup-fcc-advanced-{stamp}"
SCENARIO_ID = "kdd-workorder-fcc-advanced-001"
trajectory_path = TRACE_DIR / f"{RUN_ID}.json"
stdout_path = LOG_DIR / f"{RUN_ID}.stdout.log"
stderr_path = LOG_DIR / f"{RUN_ID}.stderr.log"
env = MCP_ENV.copy()
env["AGENT_TRAJECTORY_DIR"] = str(TRACE_DIR)
cmd = [
    "uv", "run", "--directory", str(REPO), "stirrup-agent",
    "--no-code", "--json", "--max-turns", "4",
    "--model-id", MODEL_ID, "--run-id", RUN_ID,
    "--scenario-id", SCENARIO_ID, QUESTION,
]
try:
    completed = subprocess.run(
        cmd, env=env, text=True, capture_output=True, timeout=AGENT_TIMEOUT_SECONDS
    )
    stdout_text, stderr_text = completed.stdout or "", completed.stderr or ""
    returncode = completed.returncode
except subprocess.TimeoutExpired as exc:
    stdout_text, stderr_text = exc.stdout or "", exc.stderr or ""
    if isinstance(stdout_text, bytes): stdout_text = stdout_text.decode(errors="replace")
    if isinstance(stderr_text, bytes): stderr_text = stderr_text.decode(errors="replace")
    returncode = None
    print(f"Agent timed out after {AGENT_TIMEOUT_SECONDS}s.")
stdout_path.write_text(stdout_text, encoding="utf-8")
stderr_path.write_text(stderr_text, encoding="utf-8")
if returncode not in (0, None):
    raise RuntimeError("Stirrup failed:\n" + "\n".join(stderr_text.splitlines()[-25:]))
if not trajectory_path.exists():
    raise RuntimeError(f"No trajectory persisted. Logs: {stdout_path}, {stderr_path}")
print("trajectory:", trajectory_path)


## 5. Audit grounding, safety, and exact output


In [ ]:
trajectory = json.loads(trajectory_path.read_text(encoding="utf-8"))
turns = trajectory.get("trajectory", {}).get("turns", [])
calls = [call for turn in turns for call in turn.get("tool_calls", [])]
call_names = [call.get("name") for call in calls]
retrieval_calls = [call for call in calls if call.get("name") == "wo__get_workorder"]
retrieval_ok = any(
    call.get("input") == {"site_id": SITE_ID, "wonum": WORK_ORDER_NUMBER}
    and not json.loads(call.get("output", "{}")).get("error")
    for call in retrieval_calls
)
write_markers = ("generate", "update", "approve", "assign", "close", "cancel", "create", "delete")
write_calls = [name for name in call_names if name and name.startswith("wo__") and any(marker in name for marker in write_markers)]
answer = str(trajectory.get("answer", "")).strip()
record_failure = record.get("failurecode") or record.get("failure_code")
answer_shape_valid = bool(answer) and "\n" not in answer and (
    answer == "NOT FOUND" or answer in ALLOWED_DESCRIPTIONS or bool(record_failure)
)
audit = {
    "actual_tool_calls": call_names,
    "retrieval_grounded": retrieval_ok,
    "write_calls": write_calls,
    "final_answer": answer,
    "answer_shape_valid": answer_shape_valid,
    "run_valid": retrieval_ok and not write_calls and answer_shape_valid and answer != "NOT FOUND",
}
print(json.dumps(audit, indent=2))
if not audit["run_valid"]:
    raise AssertionError("RUN REJECTED: do not use this trajectory for evaluation or the leaderboard.")
print("RUN ACCEPTED:", answer)
